# 02 — Preprocessing
Clean the data, select features, encode labels, scale, and split into train/test sets.

In [ ]:
# Cell 1 — Imports
import os
import numpy as np
import pandas as pd
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
# Cell 2 — Load primary training data
df = pd.read_csv("../data/SpotifySongs.csv")
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

In [ ]:
# Cell 3 — Filter to 10 target genres
TARGET_GENRES = [
    'hip-hop', 'pop', 'rock', 'classical', 'country',
    'electronic', 'r-n-b', 'jazz', 'metal', 'comedy'
]

df = df[df['track_genre'].isin(TARGET_GENRES)].copy()
print(f"Rows after genre filter : {len(df):,}")
print(f"Genres kept             : {sorted(df['track_genre'].unique())}")

In [ ]:
# Cell 3 — Clean: drop duplicates and null rows
before = len(df)

df = df.drop_duplicates()
dupes_removed = before - len(df)

df = df.dropna()
nulls_removed = before - dupes_removed - len(df)

print(f"Duplicates removed : {dupes_removed:,}")
print(f"Null rows removed  : {nulls_removed:,}")
print(f"Rows remaining     : {len(df):,}")

In [ ]:
# Cell 4 — Feature selection: keep only audio features + target
AUDIO_FEATURES = [
    'danceability', 'energy', 'loudness', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo'
]
TARGET = 'track_genre'

df = df[AUDIO_FEATURES + [TARGET]]
print(f"Columns kept: {df.columns.tolist()}")
df.head()

In [ ]:
# Cell 5 — Encode target labels
le = LabelEncoder()
y = le.fit_transform(df[TARGET])

# Show the integer → genre mapping
print("Label encoding map:")
for i, genre in enumerate(le.classes_):
    print(f"  {i:>3} → {genre}")

In [ ]:
# Cell 6 — Scale features (fit on train, transform both splits)
X = df[AUDIO_FEATURES].values

# Train/test split first so the scaler is fit only on training data
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")

In [ ]:
# Cell 7 — Verify class distribution in each split
train_counts = pd.Series(y_train).value_counts().sort_index()
test_counts  = pd.Series(y_test).value_counts().sort_index()

dist = pd.DataFrame({
    'genre'      : le.classes_,
    'train_count': train_counts.values,
    'test_count' : test_counts.values
})
print(dist.to_string(index=False))

In [ ]:
# Cell 8 — Save processed arrays, scaler, and encoder
os.makedirs("../data/processed", exist_ok=True)
os.makedirs("../models", exist_ok=True)

np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/X_test.npy",  X_test)
np.save("../data/processed/y_train.npy", y_train)
np.save("../data/processed/y_test.npy",  y_test)

joblib.dump(scaler, "../models/scaler.pkl")
joblib.dump(le,     "../models/label_encoder.pkl")

print("Saved:")
print("  data/processed/X_train.npy")
print("  data/processed/X_test.npy")
print("  data/processed/y_train.npy")
print("  data/processed/y_test.npy")
print("  models/scaler.pkl")
print("  models/label_encoder.pkl")